# ESA Mediterranean Sea Datacube

The ESA Mediterranean Sea Datacube brings selected marine datasets onto a common 1/24° latitude/longitude grid (EPSG:4326) for the Mediterranean Sea.

The data are available in four online GeoZarr stores: monthly fields, hourly currents, daily surface fields, and daily biophysics. The stores share the same horizontal grid and keep the time and depth coordinates each dataset needs.

This shared grid lets you compare variables without repeating the spatial alignment.

### Content

Use these notebooks to access the datacube, build a small example, and explore changes through time and depth.

1. [Accessing the ESA Mediterranean Sea Datacube](1_remote_cube_access.ipynb) opens each online store with `open_zarr` when needed, selects a region, and plots example layers.
2. [Build one local datacube for a small ROI](2_build_single_cube_demo.ipynb) selects salinity from the original 4DMED-SEA source dataset, resamples a small region, and saves and plots the local cube.
3. [Time Series at Every Depth](3_large_area_processing.ipynb) plots sea-water temperature through time and depth at one location.

The access examples open the online GeoZarr stores linked below directly from EarthCODE object storage. An internet connection is required; xarray reads the selected chunks without downloading the full cubes.

### Visualization

A simple visualisation of the data cube is available at https://sunnydean.github.io/ocean_cubes_vis/:

The access notebook includes maps of salinity, sea level and currents, phytoplankton and carbonate chemistry, primary production and temperature, and hourly currents. Use the [individual dataset notebooks](../1_Datasets/datasets_sumary.ipynb) to explore the source products.

## Data and access

- All spatial layers are aligned to one EPSG:4326 `lat`/`lon` grid. Full-resolution level `0` uses 1/24° cells, with longitude centres from approximately `-6.0625` to `36.0625` and latitude centres from `30.2708` to `45.9792`, giving 1,012 columns by 378 rows.
- Each store is a multiscale GeoZarr with groups `0`, `1`, and `2`. These represent 1/24°, 1/6°, and 1/3° spacing, with spatial shapes 378 × 1,012, 94 × 253, and 47 × 126.
- The same `lat, lon` position refers to the same place across every store at a given level. A degree grid has different physical cell areas at different latitudes.
- Source fields are spatially resampled using **nearest neighbour**
- Monthly fields are averaged into calendar months and indexed by month start. Daily fields are aligned to daily timestamps.
- The biophysics cube preserves 18 common depth levels from 3 to 135 m, positive down. WOC has one depth level at 15 m. Select dates and depths that contain data for the variables you use.
- All four stores include `water_mask`: `1` retains water or unmapped offshore cells and `0` excludes mapped land. It comes from ESA WorldCover 2021 overviews and is applied across time and depth; The mask is not guaranteed permanent water, but rather an np.isin([0,80]) (i.e. non land and permanent water bodies) applied to the land classificaiton product.

### Table of Cubes

The access links below point to the online GeoZarr stores. Dimensions and coordinate coverage describe the supplied stores, rather than guaranteeing a valid observation at every cell and date.

| Cube | Access | Datasets | Dimensions at level 0 | Coordinate coverage | Licence |
|---|---|---|---|---|---|
| Monthly fields | [ocean-med-monthly-cube.zarr](https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/med_cubes/ocean-med-monthly-cube.zarr) | 4DMED PFT/Kd; Atlantic Ocean heat content; OceanSODA monthly and 8-day products; BICEP; MITHO | `time: 84`, `lat: 378`, `lon: 1,012` | January 2016–December 2022, monthly | Mixed; see source terms below |
| Hourly currents | [hourly_cube.zarr](https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/med_cubes/hourly_cube.zarr) | WOC total, ageostrophic, and CMEMS geostrophic/total current components | `time: 43,800`, `depth: 1`, `lat: 378`, `lon: 1,012` | 30 December 2014 00:00–30 December 2019 23:00, hourly; 15 m depth | CC BY 4.0 |
| Daily biophysics | [ocean-med-biophysics.zarr](https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/med_cubes/ocean-med-biophysics.zarr) | 4DMED primary production and 3D physical fields | `time: 2,404`, `depth: 18`, `lat: 378`, `lon: 1,012` | 1 January 2016–31 July 2022, daily; 3–135 m depth | CC BY 4.0 |
| Daily surface fields | [ocean-med-daily-surface-cube.zarr](https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/med_cubes/ocean-med-daily-surface-cube.zarr) | 4DMED salinity/density, MIOST FSLE, both 4DVarNet products; CAREHeat with and without SSA | `time: 2,557`, `lat: 378`, `lon: 1,012` | 1 January 2016–31 December 2022, daily | CC BY 4.0 |

<!-- ### Source Products and Cube Variables

The OSC links below are the catalogue entries listed in the repository README. The access notebooks describe the prepared source assets; the variable names here describe the combined cubes.

| Store | Source product | Cube variables | Description and combination |
|---|---|---|---|
| Monthly | [4DMED PFT and Kd - OSC](https://opensciencedata.esa.int/products/4dmed-2d-pft-kd/collection) · [Notebook](../1_Datasets/4dmed-2d-pft-kd/access.ipynb) | `DIATO`, `DINO`, `GREEN`, `HAPTO`, `PROKA`; `Kd_blue`, `Kd_UVA`, `Kd_UVAB` | Five phytoplankton chlorophyll-a groups and three attenuation bands; PFT and Kd assets share the monthly grid. Source coverage is 2019–2021. CC BY 4.0. |
| Monthly | [Atlantic Ocean heat content - OSC](https://opensciencedata.esa.int/products/4d-atlantic-ohc-global/collection) · [Notebook](../1_Datasets/oceah-heat/access.ipynb) | `ohc` | Heat content per unit area from the coarse source grid. The workflow drops `ohc_mask`; neither it nor source `cell_surface` is in the supplied store. AVISO terms. |
| Monthly | [OceanSODA-ETHZ - OSC](https://opensciencedata.esa.int/products/ocean-soda-ethz/collection) · [Notebook](../1_Datasets/ocean-soda/access.ipynb) | `ph_total`, `ph_free`, `dic`, `talk`, `spco2`, `sfco2`, `omega_ar`, `omega_ca`, `co2`, `co3`, `hco3`, `revelle_factor`, `temperature`, `salinity`, and uncertainty fields | Monthly carbonate chemistry is combined with monthly means of 8-day `dfco2`, `fgco2`, `ice`, `kw`, `sol`, and related products. Colliding names retain suffixes such as `temperature_oceansoda_8day_temperature` and `fgco2_oceansoda_8day_fgco2_carbontracker`. CC BY-NC-SA 4.0. |
| Monthly | [BICEP - OSC](https://opensciencedata.esa.int/products/bicep-database/collection.json) · [Notebook](../1_Datasets/bicep/access.ipynb) | `pp`; `EP_Dunne`, `EP_Henson`, `EP_Li`; `POC`, `POC_bias`, `POC_rmsd`, `Rrs_490`, `Rrs_560`; `C_microphyto`, `C_nanophyto`, `C_phyto`, `C_picophyto`, `chl_a`, `mean_spectral_i_star`, `mld`, `par` | Primary productivity, export production, particulate organic carbon, and phytoplankton carbon assets are combined by month. Source coverage ends in 2020. UK Open Government Licence. |
| Monthly | [MITHO - OSC](https://opensciencedata.esa.int/products/global-cumulative-hazard-indexes-chis/collection.json) · [Notebook](../1_Datasets/mitho/access.ipynb) | `CH1_CHI1`, `CH3_CHI3`, `CHI2`, `CHI4`, stressors, normalized and moving-average variants, and supplied diagnostics | CHI2 uses source `level=0`. Shared CHI4 stressor names gain `_mitho_chi4`. Category-count layers are split into `_low`, `_medium`, `_high` and repeated along cube time; they are not monthly event categories. The README lists CC BY-SA 4.0, while the configs' embedded OSC collections list CC BY 4.0; verify the source terms before reuse. |
| Hourly | [WOC currents at 15 m - OSC](https://opensciencedata.esa.int/products/total-surface-current-15/collection) · [Notebook](../1_Datasets/woc/access.ipynb) | `ut_woc`, `vt_woc`, `ue_woc`, `ve_woc`; `ut_cmems`, `vt_cmems`, `ue_cmems`, `ve_cmems`, `ug_cmems`, `vg_cmems` | Eastward/northward total, ageostrophic, and geostrophic components; spatial nearest-neighbour resampling preserves the hourly axis. CC BY 4.0. |
| Biophysics | [4DMED primary production - OSC](https://opensciencedata.esa.int/products/4dmed-3d-prim-prod-150/collection) · [Notebook](../1_Datasets/4dmed-pp/access.ipynb) | `PP`, `ZEU` | Volumetric primary production and euphotic depth. Negative source depths are converted to positive-down depths and 18 levels common to the physical fields are selected. CC BY 4.0. |
| Biophysics | [4DMED 3D physical fields - OSC](https://opensciencedata.esa.int/products/4dmed-t-s-geo-150/collection) · [Notebook](../1_Datasets/4dmed-t-s-geo-150/access.ipynb) | `to`, `so`, `ugo`, `vgo`, `zo`, `mlotst` | Temperature, salinity, geostrophic currents, absolute height, and mixed-layer depth. Native depth levels are selected and times are aligned by day; `ZEU` and `mlotst` remain depth-independent. CC BY 4.0. |
| Daily surface | [4DMED sea-surface salinity - OSC](https://opensciencedata.esa.int/products/4dmed-2d-sss/collection) · [Notebook](../1_Datasets/4dmed-ss/access.ipynb) | `sos`, `sos_error`, `dos`, `dos_error` | Salinity, density, and their error fields; source `depth=0` is selected and dropped. CC BY 4.0. |
| Daily surface | [4DMED MIOST Lagrangian eddies - OSC](https://opensciencedata.esa.int/products/4dmed-2d-alt-miost-le-24/collection) · [Notebook](../1_Datasets/4dmed-fsle/access.ipynb) | `fsle` | Backward finite-size Lyapunov exponents, aligned spatially and by day. Source coverage starts in April 2016. CC BY 4.0. |
| Daily surface | [4DVarNet 1/8° - OSC](https://opensciencedata.esa.int/products/4dmed-2d-alt-varnet-8/collection) · [Notebook](../1_Datasets/4dmed-4dvar-8/acces.ipynb) | `adt_4dvarnet`, `sla_4dvarnet`, `ugos_4dvarnet`, `vgos_4dvarnet`, `ugosa_4dvarnet`, `vgosa_4dvarnet`, `relative_vorticity_4dvarnet` | Sea level, absolute and anomalous geostrophic currents, and relative vorticity. Product suffixes distinguish the two reconstructions. CC BY 4.0. |
| Daily surface | [4DVarNet 1/20° - OSC](https://opensciencedata.esa.int/products/4dmed-2d-alt-varnet-20/collection) · [Notebook](../1_Datasets/4dmed-4dvar-20/acces.ipynb) | The same seven fields with `_4dvarnet_ssh` suffixes | The finer source reconstruction is resampled separately onto the common grid. CC BY 4.0. |
| Daily surface | [CAREHeat - OSC](https://opensciencedata.esa.int/projects/careheat/collection) · [Notebook](../1_Datasets/careheat/access.ipynb) | `category`, `ssa_category` | Daily marine-heatwave categories without and with SSA; nearest-neighbour resampling preserves class values. Spatial summaries use maximum. CC BY 4.0. |

The separate `ocean-cube.json` prototype contains `CHL3D`, `DS3D`, `SAL3D`, `TEMP3D`, `UO3D`, and `VO3D` from the 3D biophysical product. It is not the supplied `ocean-med-biophysics.zarr` store. WAPOSAL and the other catalogue products absent from the four build workflows remain available as individual datasets. -->

### Current Assumptions and Future Refinements

Upsampling or downsampling changes how observations are represented and can make a cube unsuitable for some scientific analyses. The 1/24° output grid does not create fine-scale information in a 1° source. The original prepared products remain available through [1_Datasets](../1_Datasets/datasets_sumary.ipynb) for workflows that need their native grids, times, or depth levels.

Monthly aggregation of 8-day products uses an unweighted mean of samples assigned to each calendar month; it is not a day-overlap-weighted integration. Reindexed dates may be entirely missing for some variables. Source-wide MITHO summaries repeated along time should not be treated as independent monthly observations. Likewise, a maximum category in a coarser GeoZarr group represents the most severe contributing cell, rather than a class at every location in that block.

Open the store containing the variables you need, then select a region, time, and depth before computing. When comparing hourly, daily, and monthly values, aggregate them to a common time interval.